In [12]:
!pip install transformers datasets torch accelerate

In [13]:
%%writefile stories.txt
A mysterious letter arrived at the boy’s doorstep one rainy evening.
Deep beneath the ocean, a forgotten city slowly began to wake up.
Every night at midnight, the clock in the old house started ticking backwards.
A small dragon hatched in the most unexpected place, a school backpack.
The traveler opened a door that shouldn’t have existed in the middle of the desert.
In a quiet town, shadows started moving even when no one was there.

Overwriting stories.txt


In [14]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

model = GPT2LMHeadModel.from_pretrained("gpt2")

print("Model loaded successfully!")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully!


In [15]:
from datasets import load_dataset

dataset = load_dataset("text", data_files={"train": "stories.txt"})

print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 6
    })
})


In [16]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenizer.pad_token = tokenizer.eos_token
tokenized_data = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

In [17]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [18]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./gpt2_story_model",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=2,
    logging_steps=100,
    fp16=True
)

In [19]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data["train"],
    data_collator=data_collator,
)

trainer.train()

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=9, training_loss=4.308090633816189, metrics={'train_runtime': 15.8349, 'train_samples_per_second': 1.137, 'train_steps_per_second': 0.568, 'total_flos': 1175814144000.0, 'train_loss': 4.308090633816189, 'epoch': 3.0})

In [20]:
model.save_pretrained("./my_story_model")
tokenizer.save_pretrained("./my_story_model")

print("Model Saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model Saved!


In [21]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("./my_story_model")
model = GPT2LMHeadModel.from_pretrained("./my_story_model")

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

prompt = "Once upon a time"

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    padding=True,
    truncation=True
)

input_ids = inputs["input_ids"].to(model.device)
attention_mask = inputs["attention_mask"].to(model.device)

output = model.generate(
    input_ids,
    max_length=200,
    temperature=0.8,
    top_k=50,
    top_p=0.9,
    do_sample=True
)

story = tokenizer.decode(output[0], skip_special_tokens=True)

print("Generated Story:\n")
print(story)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Generated Story:

Once upon a time, in the very first month of his presidency, Bush was elected president.

On Jan. 20, 2001, he was sworn in as the new president of the United States.

At the height of his power, his campaign manager, John Podesta, had a very clear vision of the nation, a vision of what he could do with his time.

"I think this is a great opportunity to build upon that legacy of building a nation of our own, and build a country for the future of the world," he told The Washington Post, referring to his first term.

He is also very active in his political campaigns.

He has appeared on television and in the media in the past. His first national ad in 2004 was called "The World's Greatest National Day" in which he called on Americans to donate $1,000 to the Clinton Foundation, a non-profit foundation he was appointed to help.

He was also elected
